<a href="https://colab.research.google.com/github/nikhil00shinde/ml/blob/main/Day001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install numpy pandas matplotlib

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("kc_house_data.csv");

In [20]:
print("━━━━ SHAPE ━━━━")
print(df.shape)

print("\n━━━━ COLUMN TYPES ━━━━")
print(df.dtypes)

print("\n━━━━ FIRST 5 ROWS ━━━━")
print(df.head())

print("\n━━━━ MISSING VALUES ━━━━")
print(df.isnull().sum())


print("\n━━━━ STATISTICAL SUMMARY ━━━━")
print(df.describe())

print(df.columns)


━━━━ SHAPE ━━━━
(21613, 21)

━━━━ COLUMN TYPES ━━━━
id                 int64
date              object
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object

━━━━ FIRST 5 ROWS ━━━━
           id             date     price  bedrooms  bathrooms  sqft_living  \
0  7129300520  20141013T000000  221900.0         3       1.00         1180   
1  6414100192  20141209T000000  538000.0         3       2.25         2570   
2  5631500400  20150225T000000  180000.0         2       1.00          770   
3  2487200875  20141209T000000  604000.0         4       3.00         196

# ML Engineering Institute — Complete Study Notes
---

## Progress Tracker

| Day | Topic | Status |
|-----|-------|--------|
| Day 0 | Python rules: return vs print, `//`, mutables, type hints | ✅ Done |
| Day 0 | Built `describe()` — 3 iterations from scratch | ✅ Done |
| Day 0 | NumPy: vectorization, boolean indexing, aggregations | ✅ Done |
| Day 0 | NumPy: `np.sort()` vs `sorted()` — stay in NumPy world | ✅ Done |
| Day 0 | Stats: mean, median, std — with geometric intuition | ✅ Done |
| Day 0 | Stats: Q1, Q3, IQR, outlier detection (1.5×IQR rule) | ✅ Done |
| Day 0 | Stats: right skew, left skew, log transform preview | ✅ Done |
| Day 1 | 5-line EDA ritual on 21,613 row real dataset | ✅ Done |
| Day 1 | Read dtypes — found date (string), zipcode (int) errors | ✅ Done |
| Day 1 | Found 33-bedroom anomaly in `describe()` output | ✅ Done |
| Day 1 | Understood `yr_renovated=0` encoding problem + fix | ✅ Done |
| Day 1 | Understood `sqft_living` vs `sqft_living15` relationship | ✅ Done |
| Day 2 | Data cleaning + Matplotlib visualizations | ⬜ Next |

---

# DAY 0 — Python & NumPy Foundations

> Everything in Day 0 was written from scratch — no ML libraries, pure Python.
> Goal: build correct thinking habits before touching any tooling.

---

## 1. Python — The 4 Rules That Matter

### Rule 1 — Functions Return, They Don't Print

> **A function that prints is a dead end. You cannot test it, chain it, or use its output anywhere in a pipeline.**

In ML pipelines you call `describe()` and feed its output into the next step. Printing breaks that entirely.

```python
# ❌ WRONG — prints and returns nothing useful
def describe_bad(data) -> None:
    print("Mean:", sum(data)/len(data))   # dead end — can't use this anywhere

# ✅ CORRECT — returns a dict, caller decides what to do
def describe_good(data: list) -> dict:
    return {'mean': sum(data) / len(data)}

prices = [213000, 538000, 180000, 604000, 320000]
result = describe_good(prices)  # now usable
mean   = result['mean']         # chainable
print(f"Mean: {mean:,.0f}")     # print at the CALL SITE, not inside the function
```

---

### Rule 2 — Integer Division with `//`

`int(x/2)` creates a float first, then converts. `//` is direct integer division — Pythonic and faster.

```python
n = 10
mid_wrong   = int(n / 2)   # ❌ float created first, then truncated
mid_correct = n // 2       # ✅ direct integer division — always use this
```

---

### Rule 3 — Mutable Objects Are Shared, Not Copied

This bites people hard when manipulating DataFrames and wondering why their original data changed.

```python
# The trap
a = [1, 2, 3]
b = a           # b IS a — same object in memory, NOT a copy
b.append(4)
print(a)        # [1, 2, 3, 4] ← surprise!

# The fix
a = [1, 2, 3]
b = a.copy()    # now b is a truly independent copy
b.append(4)
print(a)        # [1, 2, 3] ← safe
```

---

### Rule 4 — Type Annotations Are Contracts

When you write `-> None` but return a dict, you broke the contract before writing a single line of logic.

```python
# Contract: takes a list, returns a dict
def describe(data: list) -> dict:    # <- this is a promise to every reader
    n = len(data)
    return {'count': n, 'mean': sum(data) / n}
```

---

## 2. Building `describe()` — 3 Iterations

We built this function from scratch, improving it each round.

| Version | What Was Wrong | What We Fixed |
|---------|----------------|---------------|
| v1 | Printed instead of returned. Wrong `-> None` annotation | Return dict, fix annotation |
| v2 | No median. Calling `min()`/`max()` twice each (wasted scans) | Added median with even/odd logic, store min/max once |
| v3 | Pure Python — slow on large data | Full NumPy rewrite with std, q1, q3, iqr |

### Version 1 — What You First Submitted (Broken)

```python
def describe(data) -> None:           # ❌ wrong return type
    n = len(data)
    print("Count:", n)                # ❌ prints — dead end
    print("Mean:", sum(data)/n)
    print("Min:", min(data))          # ❌ scans full list
    print("Max:", max(data))          # ❌ scans full list again
    print("Range:", max(data) - min(data))  # ❌ two more full scans
```

### Version 2 — Pure Python, Correct Logic

```python
def describe(data: list) -> dict:
    n           = len(data)
    total       = sum(data)
    sorted_data = sorted(data)  # one sort, reuse it
    minimum     = sorted_data[0]
    maximum     = sorted_data[-1]
    mid         = n // 2        # ✅ integer division

    # Even list: average the two middle values
    # Odd list: take the exact middle value
    if n % 2 == 0:
        median = (sorted_data[mid - 1] + sorted_data[mid]) / 2
    else:
        median = sorted_data[mid]

    return {
        'count' : n,
        'mean'  : total / n,
        'median': median,
        'min'   : minimum,
        'max'   : maximum,
        'range' : maximum - minimum
    }

prices = [213000, 538000, 180000, 604000, 320000]
result = describe(prices)

# Print at the CALL SITE — not inside the function
for key, value in result.items():
    print(f"{key:>10} : {value:,.2f}")
```

### Version 3 — NumPy (Final Version)

```python
import numpy as np

def describe(data: list) -> dict:
    arr  = np.array(data)
    n    = len(arr)
    mid  = n // 2
    sarr = np.sort(arr)          # np.sort stays in NumPy world

    median = (sarr[mid-1] + sarr[mid]) / 2 if n % 2 == 0 else sarr[mid]
    q1     = np.percentile(arr, 25)
    q3     = np.percentile(arr, 75)

    return {
        'count' : n,
        'mean'  : arr.mean(),
        'median': median,
        'std'   : arr.std(),
        'min'   : arr.min(),
        'max'   : arr.max(),
        'range' : arr.max() - arr.min(),
        'q1'    : q1,
        'q3'    : q3,
        'iqr'   : q3 - q1
    }

prices = [213000, 538000, 180000, 604000, 320000,
          750000, 289000, 415000, 560000, 198000]
result = describe(prices)
for key, value in result.items():
    print(f"{key:>10} : {value:,.2f}")
```

---

## 3. NumPy — Core Mental Models

### Why NumPy Exists

> **Python lists store pointers to objects scattered in memory. NumPy stores data in a contiguous block — same type, same size, packed together. Operations run in C. Result: ~100x faster.**

```python
# Python list — jumps around memory, slow
doubled_slow = [p * 2 for p in prices_list]   # Python loop

# NumPy — contiguous memory, C speed
prices_arr   = np.array(prices_list)
doubled_fast = prices_arr * 2                  # vectorized — no Python loop
```

This is called **vectorization** — and it's the mental model behind everything in Pandas, PyTorch, and every ML framework.

---

### Creating Arrays — The 5 Ways

```python
import numpy as np

a = np.array([1, 2, 3, 4, 5])        # from list
b = np.zeros(5)                        # [0. 0. 0. 0. 0.]
c = np.ones((3, 3))                    # 3x3 matrix of 1s
d = np.arange(0, 10, 2)              # [0 2 4 6 8]
e = np.linspace(0, 1, 5)            # [0. 0.25 0.5 0.75 1.]
```

---

### Shape & Dtype — Always Know These

```python
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

print(arr.shape)   # (2, 3) — 2 rows, 3 cols
print(arr.dtype)   # int64
print(arr.ndim)    # 2 — number of dimensions
print(arr.size)    # 6 — total number of elements
```

---

### Vectorized Operations — No Loops

```python
prices = np.array([213000, 538000, 180000, 604000, 320000])

print(prices * 1.1)              # apply 10% increase to ALL elements
print(prices - prices.mean())   # center the data around zero
print(prices > 300000)          # boolean array — True/False per element
```

---

### Boolean Indexing — How You Filter Data

> **This is the exact mechanism Pandas uses internally for `.loc` filtering. Master this.**

```python
prices = np.array([213000, 538000, 180000, 604000, 320000])

# Step 1: create a boolean mask
mask = prices > 300000
print(mask)             # [False  True False  True  True]

# Step 2: use mask to filter
print(prices[mask])     # [538000 604000 320000]

# One-liner (most common in practice)
print(prices[prices > 300000])

# Compound conditions — use & (and) | (or)
mid_range = prices[(prices > 200000) & (prices < 600000)]
```

---

### Aggregations

```python
prices = np.array([213000, 538000, 180000, 604000, 320000])

print(prices.mean())                 # average
print(prices.std())                  # standard deviation
print(prices.min(), prices.max())   # min and max
print(np.percentile(prices, 25))    # Q1 — 25th percentile
print(np.percentile(prices, 75))    # Q3 — 75th percentile
print(np.sort(prices))              # sorted array
```

---

### `np.sort()` vs `sorted()` — Stay in NumPy World

> **Once you convert data to NumPy, stay in NumPy. Every conversion back to Python costs time.**

| | `sorted()` | `np.sort()` |
|--|--|--|
| Works on | Any iterable | NumPy arrays |
| Returns | Python **list** | NumPy **array** |
| Speed | Slow (Python) | Fast (C) |
| Rule | Use with lists | Use with arrays |

```python
arr = np.array([3, 1, 4, 1, 5])

sorted(arr)      # ❌ converts back to Python list — left NumPy world
np.sort(arr)     # ✅ stays in NumPy world — always prefer this
```

---

## 4. Statistics — What Everyone Memorizes But Nobody Understands

### Mean vs Median

**Mean** is the balance point — every value pulls it toward itself.
**Median** is the middle value — resistant to outliers.

```
# Why housing data always reports median price, never mean:

prices = [198k, 213k, 289k, 320k, 415k, 538k, 560k, 604k, 750k, 999k]
mean   = 488,600   ← mansion at 999k pulls this rightward
median = 476,500   ← doesn't care about the mansion at all
```

> **Rule: Right skew → mean > median | Left skew → mean < median | Symmetric → mean ≈ median**

---

### Skewness — Visualized

```
RIGHT SKEW (housing prices)         LEFT SKEW (exam scores)

  ▓                                              ▓
  ▓▓▓                                        ▓▓▓▓▓
  ▓▓▓▓▓▓                                ▓▓▓▓▓▓▓▓▓
  ▓▓▓▓▓▓▓▓▓▓░░░░░░░░░       ░░░░░░░░░▓▓▓▓▓▓▓▓▓▓▓▓▓
  ─────────────────────      ─────────────────────────
    ↑median  ↑mean              mean↑    median↑

Tail stretches RIGHT            Tail stretches LEFT
```

**Why it matters for ML:**

```python
# A skewed price column confuses models — fix it with log transform
df['price_log'] = np.log(df['price'])

# Before: 200k,  450k,  7,700,000   ← huge gap on right
# After:  12.2,  13.0,     15.9     ← compressed, more balanced
```

---

### Standard Deviation — Average Distance From The Mean

```python
prices = np.array([198000, 213000, 289000, 320000, 415000,
                   538000, 560000, 604000, 750000, 999000])

mean = prices.mean()   # 488,600
std  = prices.std()    # ~243,000

# Interpretation:
# "The typical house is ~$243,000 away from the average price"
# Typical range = mean ± std = $245,600 → $731,600

# High STD → prices all over the place (mixed neighborhood)
# Low  STD → prices clustered tight (uniform neighborhood)
```

---

### Q1, Q3, IQR — The Middle 50%

```
         Q1                  Median                 Q3
          ↓                     ↓                   ↓
198k | 213k | 289k | 320k || 415k | 538k || 560k | 604k | 750k | 999k
 \_________25%_________/    \____50%____/    \__________75%__________/
```

- **Q1** = 25th percentile → 25% of houses cost less than this
- **Q3** = 75th percentile → 75% of houses cost less than this
- **IQR** = Q3 - Q1 → width of the middle 50%

```python
q1  = np.percentile(prices, 25)
q3  = np.percentile(prices, 75)
iqr = q3 - q1
```

---

### Outlier Detection with IQR (1.5× Rule)

```python
# Used in every ML preprocessing pipeline
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Anything outside these fences = suspected outlier
outliers = prices[(prices < lower_fence) | (prices > upper_fence)]
```

---

### Statistics Summary Table

| Stat | What It Measures | Outlier Sensitive? | Use When... |
|------|-----------------|-------------------|-------------|
| Mean | Balance point | ✅ Yes | Data is symmetric |
| Median | Middle value | ❌ No | Data is skewed (housing) |
| Std | Avg distance from mean | ✅ Yes | Understanding spread |
| Q1 / Q3 | Boundary of middle 50% | ❌ No | Understanding typical range |
| IQR | Width of middle 50% | ❌ No | Detecting outliers |

---

---

# DAY 1 — Pandas EDA on King County Housing

> **Dataset:** 21,613 house sales in King County, WA | 21 columns | 0 missing values
> **Goal:** Read the data like a story — find every anomaly before touching any model.

---

## 1. The 5-Line EDA Ritual — Memorize This Forever

> **Run these 5 lines on every new dataset, without exception.**

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('kc_house_data.csv')

print(df.shape)           # How big is it?
print(df.dtypes)          # Are columns the right type?
print(df.head())          # Does the data look sane at first glance?
print(df.isnull().sum())  # Any missing values?
print(df.describe())      # Anything statistically impossible?
```

| Line | Question You're Asking |
|------|------------------------|
| `df.shape` | How many rows and columns do I have? |
| `df.dtypes` | Are dates stored as strings? Numbers as objects? |
| `df.head()` | Does the data look sane at first glance? |
| `df.isnull().sum()` | Where is data missing — and how much? |
| `df.describe()` | Does anything look statistically impossible? |

---

## 2. What The Output Told Us

### Shape: (21613, 21)

21,613 houses. 21 columns. Fast to run, rich enough to learn from. Perfect training dataset.

---

### Missing Values: All Zero

Clean dataset — no missing values anywhere. Rare in the real world.
Titanic (Phase 2) will show what missing data pain actually feels like.

---

### Type Errors Found in `df.dtypes`

| Column | Stored As | Problem | Fix |
|--------|-----------|---------|-----|
| `date` | object (string) | Can't sort, extract month/year, compute differences | Convert to datetime |
| `zipcode` | int64 | Model thinks 98199 > 98001 meaningfully — wrong | Convert to category |
| `waterfront` | int64 | Actually categorical — 0 or 1 only | One-hot encode later |
| `yr_renovated` | int64 | 0 means 'never renovated', not 'year 0 AD' | Engineer `was_renovated` |

---

### Anomalies Found in `df.describe()`

| Column | Anomaly | What It Means | Action |
|--------|---------|---------------|--------|
| `price` | mean 540k, median 450k | Right-skewed — mansions pulling mean up | Log transform before modeling |
| `bedrooms` | max = 33 | Data entry error — doesn't exist in KC | Find and remove this row |
| `bedrooms` | min = 0 | A house with 0 bedrooms | Investigate and remove |
| `yr_renovated` | 75% = 0, mean = 84 | 0 means not renovated, not year 0 AD | Engineer `was_renovated` binary |
| `sqft_lot` | max = 1,651,359 sqft | 37 acres — farm/estate in residential data | Investigate outlier |
| `id` | Unique identifier only | Meaningless for price prediction | Drop this column |
| `zipcode` | Stored as integer | Model treats as continuous — wrong | Convert to category |

---

## 3. Key Concepts from Day 1

### Right Skew in King County Prices

```python
# From df.describe()
mean   = 540,088   # pulled rightward by mansions
median = 450,000   # resistant to outliers
max    = 7,700,000 # the mansion pulling everything

# Mean is $90,000 ABOVE median → confirmed right skew
# Fix: apply log transform before feeding price into any model
df['price_log'] = np.log(df['price'])
```

---

### The yr_renovated Problem + Fix

```python
# The problem — model sees this:
# 0      → "renovated in year 0 AD" (for 16,000+ houses)
# 1991   → renovated in 1991
# 2015   → renovated in 2015
# Mean = 84 → model thinks "average renovation was year 84 AD" — nonsense

# The fix — convert to binary
df['was_renovated'] = (df['yr_renovated'] > 0).astype(int)
# 1 = was renovated at some point
# 0 = never renovated
```

---

### The Neighborhood Effect

```python
# sqft_living      = this specific house's size
# sqft_living15    = average size of 15 nearest neighbor houses

# Together they reveal something neither column alone can tell you:
df['size_vs_neighbors'] = df['sqft_living'] / df['sqft_living15']

# > 1.0 → biggest house on the block
# < 1.0 → smallest house on the block
# = 1.0 → average for the area

# SAME 1,500 sqft house sells for different prices depending on neighbors:
# Surrounded by 800 sqft houses  → commands premium
# Surrounded by 2,500 sqft houses → sells at discount
# This single engineered feature will improve model accuracy
```

---

### Fractional Bathrooms — Not a Bug

```
US real estate convention:
1.00 = full bath     (toilet + sink + shower + tub)
0.75 = 3/4 bath      (toilet + sink + shower)
0.50 = half bath     (toilet + sink only)

So: 2.25 bathrooms = 2 full baths + 1 half bath
    2.75 bathrooms = 2 full baths + 1 three-quarter bath
```

---

## 4. Day 2 Fix List — Everything We Need to Clean

```
⬜ Fix date column       → convert to datetime, extract month and year
⬜ Fix zipcode           → convert from int to category type
⬜ Remove bedrooms == 33 → data entry error
⬜ Remove bedrooms == 0  → data entry error
⬜ Engineer was_renovated     → (yr_renovated > 0).astype(int)
⬜ Engineer size_vs_neighbors → sqft_living / sqft_living15
⬜ Drop id column        → useless for prediction
⬜ Histogram             → visualize price distribution, confirm right skew
⬜ Boxplot               → see outliers visually
⬜ Scatter plot          → sqft_living vs price, see the relationship
⬜ Heatmap               → correlation matrix across all columns
```

---

---

# Quick Reference

## NumPy Cheatsheet

```python
import numpy as np

# Create
np.array([1,2,3])          # from list
np.zeros(5)                 # all zeros
np.ones((3,3))              # all ones
np.arange(0, 10, 2)        # [0,2,4,6,8]
np.linspace(0, 1, 5)      # 5 evenly spaced points

# Inspect
arr.shape                   # dimensions
arr.dtype                   # data type
arr.ndim                    # number of dimensions

# Operate (vectorized — no loops)
arr * 2                     # multiply all elements
arr + 100                   # add to all elements
arr > 300000                # boolean mask

# Filter
arr[arr > 300000]           # boolean indexing
arr[(arr > 200) & (arr < 600)]  # compound condition

# Aggregate
arr.mean()                  # average
arr.std()                   # standard deviation
arr.min(), arr.max()        # min and max
np.percentile(arr, 25)      # Q1
np.percentile(arr, 75)      # Q3
np.sort(arr)                # sorted (stays numpy)
```

---

## Pandas EDA Cheatsheet

```python
import pandas as pd

df = pd.read_csv('file.csv')

# The 5-line ritual — run on every dataset
df.shape                    # (rows, cols)
df.dtypes                   # column types
df.head()                   # first 5 rows
df.isnull().sum()           # missing value counts
df.describe()               # statistical summary

# Basic selection
df['column']                # single column (Series)
df[['col1', 'col2']]        # multiple columns (DataFrame)
df[df['price'] > 500000]    # filter rows

# Value counts
df['bedrooms'].value_counts()      # frequency of each value
df['bedrooms'].value_counts(normalize=True)  # as percentages
```

---

## Statistics Quick Reference

```python
import numpy as np

arr = np.array([...])

# Central tendency
mean   = arr.mean()
median = np.median(arr)

# Spread
std = arr.std()
q1  = np.percentile(arr, 25)
q3  = np.percentile(arr, 75)
iqr = q3 - q1

# Outlier fences (1.5x IQR rule)
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = arr[(arr < lower) | (arr > upper)]

# Skew check
if mean > median:
    print("Right skew — apply log transform")
elif mean < median:
    print("Left skew")
else:
    print("Roughly symmetric")

# Log transform for right-skewed data
import pandas as pd
df['price_log'] = np.log(df['price'])
```

---

> **Remember:** Functions return, they don't print. Stay in NumPy world.
> Mean > median means right skew. Every stat claim needs a visual.
> The 5-line EDA ritual runs on every dataset, no exceptions.